# 动手练习：结果呈现与研究前沿

来自《生物医学网络科学》（Biomedical Network Science）。

每个任务先运行“准备”格，再在“你的代码”格中完成 TODO，最后运行“核对”格：打印 ✓ 表示通过，否则会给出提示。数据文件放在与本笔记本同级的 `data/` 文件夹中。

In [ ]:
# 数据文件（JupyterLite 专用；文件已在时什么也不做）
import os, js
from pyodide.http import pyfetch
_base = str(js.location.href).split("/extensions/")[0].split("/build/")[0] + "/files/practice/data/"
os.makedirs("data", exist_ok=True)
for _f in ['opentargets_disease_genes_ga.csv', 'string_multichannel_disease_genes_v12_5.csv']:
    if not os.path.exists("data/" + _f):
        _r = await pyfetch(_base + _f)
        open("data/" + _f, "wb").write(await _r.bytes())


## 任务 1：小世界性：与同规模随机网络比较

任务 1、2、4 使用 networkx 自带的空手道俱乐部网络（Zachary 1977），任务 3 使用合成 BA 网络，任务 5 使用合成接触序列，任务 6 使用真实数据。每做完一个任务，用一句话写下“这个结果说明了什么、与什么比较”，这正是 [结果呈现](reporting-frontiers.figures.html) 的基本要求。

计算空手道网络的平均聚集系数 $C$ 和平均最短路径 $L$；生成 20 个节点数、边数相同的 ER 随机图（`nx.gnm_random_graph(n, m, seed=s)`，$s=0,\dots,19$），取其最大连通分支计算 $C_\mathrm{rand}$、$L_\mathrm{rand}$；求小世界系数 $\sigma=(C/\overline{C}_\mathrm{rand})/(L/\overline{L}_\mathrm{rand})$。

要得到的变量：`C_rand`、`L_rand`（各 20 个值）、`sigma`。

期望看到：$C$ 远大于 $C_\mathrm{rand}$，$L$ 与 $L_\mathrm{rand}$ 接近，$\sigma$ 明显大于 1（见 [小世界与无标度的统计判别](network-models.identification.html)）。

In [ ]:
# 准备（直接运行）
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

G = nx.karate_club_graph()
n, m = G.number_of_nodes(), G.number_of_edges()
C = nx.average_clustering(G)
L = nx.average_shortest_path_length(G)

In [ ]:
C_rand, L_rand = [], []
for s in range(20):
    R = nx.gnm_random_graph(n, m, seed=s)
    R = R.subgraph(max(nx.connected_components(R), key=len)).copy()
    # TODO: 把 R 的平均聚集系数和平均最短路径分别加入 C_rand、L_rand
sigma = None            # TODO

In [ ]:
# 核对（不用修改）
assert len(C_rand) == len(L_rand) == 20, "应有 20 个随机图的结果"
assert np.mean(C_rand) < C / 2, "随机图的聚集系数应远小于空手道网络；检查是否用了 nx.average_clustering(R)"
assert sigma is not None and 2 < sigma < 8, "sigma 应明显大于 1（约 4）；检查公式 (C/C_rand) / (L/L_rand)，不要写反"
print("✓ 任务 1 通过")

## 任务 2：社团检测与“真实”分组的一致性

用 Louvain（`seed=0`）划分空手道网络，计算模块度 $Q$；再把划分与俱乐部的实际分裂（节点属性 `club`：`Mr. Hi` 或 `Officer`）比较：统计每个社团里两派成员各有多少，并算出“多数派之外”的节点数 `n_mixed`（每个社团中少数派人数之和）。

要得到的变量：`Q`、`n_mixed`。

期望看到：$Q$ 约 0.44；Louvain 把每一派再细分，几乎不跨派混合（见 [社团结构与社区发现](community-structure.html)）。

In [ ]:
# 准备（直接运行）
comms = nx.community.louvain_communities(G, seed=0)

In [ ]:
Q = None                # TODO: nx.community.modularity(G, comms)
n_mixed = None          # TODO: 每个社团中少数派人数之和

In [ ]:
# 核对（不用修改）
assert Q is not None and 0.35 < Q < 0.5, "Louvain 在空手道网络上的 Q 约为 0.42–0.44；检查是否调用 nx.community.modularity(G, comms)"
assert n_mixed is not None and 0 <= n_mixed <= 4, "少数派人数之和应很小（只有个别节点）；检查是否对每个社团取 min(两派人数)"
print("✓ 任务 2 通过")

## 任务 3：鲁棒性：随机失效与蓄意攻击

在 BA 网络（500 个节点）上依次删除 0%–50% 的节点，比较两种策略下最大连通分支占原网络的比例：随机删除（顺序 `order_random`）与按初始度从高到低删除（`order_attack`）。

要得到的变量：`gr`、`ga`（与 `fracs` 等长的列表）。

期望看到：随机删除时巨分支缓慢缩小；攻击枢纽时迅速崩解（见 [蓄意攻击](robustness.targeted-attack.html)）。

In [ ]:
# 准备（直接运行）
G0 = nx.barabasi_albert_graph(500, 2, seed=16)
N0 = G0.number_of_nodes()
fracs = np.linspace(0, 0.5, 11)
order_random = list(np.random.default_rng(0).permutation(N0))
order_attack = [v for v, _ in sorted(G0.degree(), key=lambda x: -x[1])]

def giant_fraction(order, f):
    H = G0.copy()
    H.remove_nodes_from(order[:int(f * N0)])
    return max((len(c) for c in nx.connected_components(H)), default=0) / N0

In [ ]:
gr, ga = [], []
# TODO: 对每个 f，分别计算 giant_fraction(order_random, f) 与 giant_fraction(order_attack, f)

In [ ]:
# 核对（不用修改）
assert len(gr) == len(ga) == len(fracs), "gr、ga 应与 fracs 等长"
assert abs(gr[0] - 1) < 1e-9 and abs(ga[0] - 1) < 1e-9, "不删节点时巨分支比例应为 1"
assert gr[4] > 0.6 and ga[4] < 0.2, "删除 20% 时，随机删除应仍保留大部分连通，攻击枢纽应使网络崩解；检查两种顺序是否用反"
print("✓ 任务 3 通过")

## 任务 4：零模型：保持度序列的随机化

任务 1 用 ER 随机图作基线，但 ER 不保留度分布。更严格的基线是度保持重连：反复随机交换两条边的端点（`nx.double_edge_swap`），度序列不变，其余结构被打乱。生成 30 个度保持随机网络（`seed = 0..29`，`nswap = 10 * 边数`，`max_tries = 10**5`），求空手道网络平均聚集系数的 $z$ 分数。

要得到的变量：`C_null`（30 个值）、`z_C`。

期望看到：$z$ 仍明显大于 2，但比与 ER 比较时小得多：一部分高聚集可由度分布解释（见 [零模型](network-construction.null-models.html)）。

In [ ]:
C_null = []
# TODO: R = G.copy(); nx.double_edge_swap(R, nswap=..., max_tries=10**5, seed=s); C_null.append(...)
z_C = None

In [ ]:
# 核对（不用修改）
assert len(C_null) == 30, "应有 30 个度保持随机网络"
assert np.mean(C_null) > np.mean(C_rand), "度保持随机网络保留了枢纽，聚集系数应高于 ER 随机图"
assert z_C is not None and z_C > 2, "z 应大于 2；检查是否每次都从 G.copy() 开始重连"
print("✓ 任务 4 通过")

## 任务 5：时间网络：时间顺序会缩小可达范围

一个合成接触序列：30 个人、60 次接触 $(u,v,t)$，时刻 $t=0,\dots,59$ 各一次。写函数 `reach_from(src)`：按时间顺序扫描接触，若接触的一端已被“感染”（且感染时刻不晚于接触时刻），另一端就被感染；返回最终被感染的人数（含源点）。再在聚合的静态网络中，用连通分支大小作为“可达人数”比较（见 [时间网络](reporting-frontiers.temporal-networks.html)）。

要得到的变量：函数 `reach_from`，`reach_temporal`、`reach_static`（长度 30 的数组，按节点编号）。

期望看到：聚合网络是连通的（每人静态可达 30 人），但沿时间一致路径平均只能到达约一半的人；聚合网络高估了传播范围。

In [ ]:
# 准备（直接运行）
rng5 = np.random.default_rng(16)
contacts = []
for t in range(60):
    u, v = rng5.choice(30, 2, replace=False)
    contacts.append((int(u), int(v), t))
Gagg = nx.Graph()
Gagg.add_nodes_from(range(30))
Gagg.add_edges_from((u, v) for u, v, _ in contacts)

In [ ]:
def reach_from(src):
    infected = {src: -1}
    # TODO: for u, v, t in sorted(contacts, key=lambda c: c[2]): 若一端已感染，另一端加入 infected
    return None

reach_temporal = None   # TODO: np.array([reach_from(i) for i in range(30)])
reach_static = None     # TODO: 每个节点所在连通分支的大小

In [ ]:
# 核对（不用修改）
_old = contacts
contacts = [(0, 1, 1), (1, 2, 2)]
assert reach_from(0) == 3 and reach_from(2) == 2, "接触 (0,1,t=1)、(1,2,t=2)：0 能到达 1 和 2；2 只能到达 1，到不了 0。检查是否按时间顺序扫描、是否要求感染时刻不晚于接触时刻"
contacts = _old
assert reach_temporal is not None and len(reach_temporal) == 30, "reach_temporal 应有 30 个值"
assert np.all(reach_temporal <= reach_static), "时间可达人数不可能超过静态网络中的连通分支大小"
assert reach_temporal.mean() < reach_static.mean(), "平均而言，时间顺序应使可达人数变小"
print("✓ 任务 5 通过")

## 任务 6：开放小分析：做一张“标题写结论”的图

数据：`data/string_multichannel_disease_genes_v12_5.csv`（724 个疾病基因之间的 STRING 关联，含 experiments、database、textmining、coexpression 四个证据通道分数）与 `data/opentargets_disease_genes_ga.csv`（疾病–基因遗传关联），来源见 [结果呈现与研究前沿的真实数据案例](reporting-frontiers.html)。

自己提出一个可以用图回答的问题，例如：四个证据通道的边数和度分布有什么不同？哪种疾病的基因在文本挖掘层中连得最紧？然后做一张图，要求：
1. 标题写结论，而不是写图的类型；
2. 有对照（另一层、另一种疾病、零模型或随机基线）；
3. 坐标轴有标签，必要时给出样本量或不确定性。

要得到的变量：`fig`（matplotlib 的 Figure）、`fig_title`（字符串，与图的标题一致）。

思考：如果只能放一张图进摘要，你会放哪一张？图注里要写清哪些分析选择（阈值、版本、零模型）？

In [ ]:
# 准备（直接运行）
import pandas as pd
mc = pd.read_csv("data/string_multichannel_disease_genes_v12_5.csv")
dg = pd.read_csv("data/opentargets_disease_genes_ga.csv")
dg = dg[dg.ga_score >= 0.7]
print(mc.head(3).to_string(index=False))

In [ ]:
fig = None
fig_title = None
# TODO: 计算你要展示的量，作图，并用 ax.set_title(fig_title) 写上结论

In [ ]:
# 核对（不用修改）
assert fig is not None and hasattr(fig, "axes") and len(fig.axes) >= 1, "fig 应是含至少一个坐标轴的 matplotlib Figure"
assert isinstance(fig_title, str) and len(fig_title) >= 8, "fig_title 应是一句写出结论的标题"
assert fig_title.strip() not in {"度分布图", "柱状图", "箱线图", "热图", "网络图", "散点图"}, "标题要写结论，而不是写图的类型"
assert any(a.get_title() == fig_title for a in fig.axes) or fig._suptitle is not None, "图上的标题应与 fig_title 一致（ax.set_title 或 fig.suptitle）"
print("✓ 任务 6 通过")